# Master Analysis Notebook: FFT vs FFT-like/Wavelet vs SVD (Java)

This notebook is a **Java-only master application** for comparing three image-analysis families across generated tile outputs:

1. **FFT**
2. **FFT-like / Wavelet**
3. **SVD methods**

It computes and compares a **unified objective** shared across methods:

$$
\text{score}=0.7\cdot(\text{mask-separation})+0.3\cdot(\text{corr-with-rec})
$$

where:

- **mask-separation** is measured using a trajectory-angle mask estimated from each tile (time-averaged single-image assumption),
- **corr-with-rec** is center-ROI correlation against reconstruction, mapped to $[0,1]$.

Outputs are saved to `notebooks/_assets/fiba_wavelet_montage/master_analysis/`.

## Technique background

### Why compare these three method families?

- **FFT methods** emphasize periodic and directional texture in the frequency domain.
- **FFT-like/Wavelet methods** provide multi-scale, localized frequency analysis.
- **SVD-based methods** preserve dominant low-rank structure and can stabilize noisy texture.

Together, these families give complementary views of tissue organization: global frequency structure, local multi-scale structure, and low-rank latent structure.

### Core math intuition (high level)

- **FFT** maps a spatial tile $I(x,y)$ into frequency space $F(u,v)$ to expose directional energy concentration.
- **Wavelet / filter-bank approaches** analyze responses across scales and orientations, capturing local directional texture where FFT can be too global.
- **SVD** factorizes image matrices as $X = U\Sigma V^\top$; large singular components preserve dominant morphology while suppressing noise-like components.

### Biological relevance

In collagen-rich tissues and related fibrous matrices, orientation and coherence are linked to biomechanical behavior and remodeling state. This notebook focuses on whether each method best captures trajectory-aligned structure while remaining consistent with reconstruction outputs.

In [1]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.io.IOException;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.ArrayList;
import java.util.Arrays;
import java.util.Comparator;
import java.util.HashMap;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Locale;
import java.util.Map;
import java.util.regex.Matcher;
import java.util.regex.Pattern;

record MethodSpec(String label, String family, String mode, String fileKey) {}
record UnifiedRow(
    String datasetDir, String base, int tileId,
    String method, String family, String sourceKind,
    double trajectoryThetaDeg, double maskSeparation, double corrWithRec, double unifiedScore
) {}
record UnifiedSummary(
    String datasetDir, String family, String method, int nTiles,
    double sepMean, double sepStd,
    double corrMean, double corrStd,
    double scoreMean, double scoreStd,
    double thetaMean
) {}

System.out.println("Java master notebook ready ✅");
System.out.println("java.version: " + System.getProperty("java.version"));

Java master notebook ready ✅
java.version: 25.0.2


In [2]:
Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    throw new RuntimeException("Could not locate project root from cwd");
}

double[][] gray01(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            out[y][x] = (0.299 * r + 0.587 * g + 0.114 * b) / 255.0;
        }
    }
    return out;
}

double[][] normalize01(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) { min = Math.min(min, v); max = Math.max(max, v); }
    double span = Math.max(1e-9, max - min);
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) out[y][x] = (a[y][x] - min) / span;
    return out;
}

double[][] convolveSame(double[][] src, double[][] kernel) {
    int h = src.length, w = src[0].length;
    int kh = kernel.length, kw = kernel[0].length;
    int ry = kh / 2, rx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = -ry; j <= ry; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -rx; i <= rx; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    s += src[yy][xx] * kernel[j + ry][i + rx];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] gaborKernel(int size, double sigma, double theta, double lambda, double gamma, double psi) {
    int r = size / 2;
    double[][] k = new double[size][size];
    for (int y = -r; y <= r; y++) {
        for (int x = -r; x <= r; x++) {
            double xr = x * Math.cos(theta) + y * Math.sin(theta);
            double yr = -x * Math.sin(theta) + y * Math.cos(theta);
            double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
            double wave = Math.cos(2.0 * Math.PI * xr / lambda + psi);
            k[y + r][x + r] = gauss * wave;
        }
    }
    return k;
}

double[][] gaborEnergy(double[][] src) {
    int h = src.length, w = src[0].length;
    int ori = 8, kSize = 13;
    double sigma = 2.2, lambda = 5.5, gamma = 0.65;
    double[][] out = new double[h][w];
    for (int oi = 0; oi < ori; oi++) {
        double theta = (Math.PI * oi) / ori;
        double[][] kRe = gaborKernel(kSize, sigma, theta, lambda, gamma, 0.0);
        double[][] kIm = gaborKernel(kSize, sigma, theta, lambda, gamma, Math.PI / 2.0);
        double[][] re = convolveSame(src, kRe);
        double[][] im = convolveSame(src, kIm);
        for (int y = 0; y < h; y++) {
            for (int x = 0; x < w; x++) {
                double mag = Math.sqrt(re[y][x] * re[y][x] + im[y][x] * im[y][x]);
                if (mag > out[y][x]) out[y][x] = mag;
            }
        }
    }
    return out;
}

double[][] morletEnergy(double[][] src) {
    return gaborEnergy(src);
}

double[][] chirpletEnergy(double[][] src) {
    int h = src.length, w = src[0].length;
    int ori = 8, size = 13;
    double sigma = 2.2, lambda = 5.5, gamma = 0.65, chirp = 0.015;
    int r = size / 2;
    double[][] out = new double[h][w];
    for (int oi = 0; oi < ori; oi++) {
        double theta = (Math.PI * oi) / ori;
        double[][] k = new double[size][size];
        for (int y = -r; y <= r; y++) {
            for (int x = -r; x <= r; x++) {
                double xr = x * Math.cos(theta) + y * Math.sin(theta);
                double yr = -x * Math.sin(theta) + y * Math.cos(theta);
                double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
                double phase = 2.0 * Math.PI * (xr / lambda + chirp * xr * xr);
                k[y + r][x + r] = gauss * Math.cos(phase);
            }
        }
        double[][] resp = convolveSame(src, k);
        for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) out[y][x] = Math.max(out[y][x], Math.abs(resp[y][x]));
    }
    return out;
}

double dominantThetaRad(double[][] img01) {
    double[][] kx = new double[][] {{-1,0,1},{-2,0,2},{-1,0,1}};
    double[][] ky = new double[][] {{-1,-2,-1},{0,0,0},{1,2,1}};
    double[][] gx = convolveSame(img01, kx);
    double[][] gy = convolveSame(img01, ky);
    int h = img01.length, w = img01[0].length;
    double jxx = 0.0, jyy = 0.0, jxy = 0.0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            jxx += gx[y][x] * gx[y][x];
            jyy += gy[y][x] * gy[y][x];
            jxy += gx[y][x] * gy[y][x];
        }
    }
    return 0.5 * Math.atan2(2.0 * jxy, jxx - jyy);
}

double[][] trajectoryMask01(int h, int w, double theta) {
    double[][] m = new double[h][w];
    double cx = (w - 1) / 2.0;
    double cy = (h - 1) / 2.0;
    double sigmaPerp = 0.10 * Math.min(h, w);
    double sigmaAlong = 0.45 * Math.min(h, w);
    double twoPerp = 2.0 * sigmaPerp * sigmaPerp;
    double twoAlong = 2.0 * sigmaAlong * sigmaAlong;
    double c = Math.cos(theta), s = Math.sin(theta);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double dx = x - cx;
            double dy = y - cy;
            double along = dx * c + dy * s;
            double perp = -dx * s + dy * c;
            m[y][x] = Math.exp(-(perp * perp) / twoPerp) * Math.exp(-(along * along) / twoAlong);
        }
    }
    return normalize01(m);
}

double maskSeparation(double[][] map01, double[][] trajMask01, double thr) {
    int h = Math.min(map01.length, trajMask01.length);
    int w = Math.min(map01[0].length, trajMask01[0].length);
    double in = 0.0, out = 0.0;
    int nin = 0, nout = 0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            if (trajMask01[y][x] >= thr) { in += map01[y][x]; nin++; }
            else { out += map01[y][x]; nout++; }
        }
    }
    double inMean = (nin == 0) ? 0.0 : in / nin;
    double outMean = (nout == 0) ? 0.0 : out / nout;
    return Math.max(0.0, Math.min(1.0, inMean - outMean));
}

double pearsonCenter(double[][] a, double[][] b) {
    int h = Math.min(a.length, b.length);
    int w = Math.min(a[0].length, b[0].length);
    int y0 = h / 4, y1 = h - h / 4;
    int x0 = w / 4, x1 = w - w / 4;
    int n = 0;
    double sa = 0.0, sb = 0.0;
    for (int y = y0; y < y1; y++) for (int x = x0; x < x1; x++) { sa += a[y][x]; sb += b[y][x]; n++; }
    if (n < 2) return 0.0;
    double ma = sa / n, mb = sb / n;
    double num = 0.0, da = 0.0, db = 0.0;
    for (int y = y0; y < y1; y++) {
        for (int x = x0; x < x1; x++) {
            double xa = a[y][x] - ma;
            double xb = b[y][x] - mb;
            num += xa * xb;
            da += xa * xa;
            db += xb * xb;
        }
    }
    return num / Math.sqrt(Math.max(1e-12, da * db));
}

Map<String, MethodSpec> buildMethodSpecs() {
    Map<String, MethodSpec> m = new LinkedHashMap<>();
    m.put("fft", new MethodSpec("FFT Pipeline", "FFT", "file", "fft"));
    m.put("gabor", new MethodSpec("Gabor Transform", "FFT-like/Wavelet", "dynamic", ""));
    m.put("chirplet", new MethodSpec("Chirplet Transform", "FFT-like/Wavelet", "dynamic", ""));
    m.put("morlet", new MethodSpec("Morlet Wavelet", "FFT-like/Wavelet", "dynamic", ""));
    m.put("svd_rank20", new MethodSpec("SVD Rank-20", "SVD", "file", "svd_rank20"));
    m.put("svd_energy99", new MethodSpec("SVD Energy-99%", "SVD", "file", "svd_energy99"));
    m.put("cnn_rank40", new MethodSpec("CNN-style ConvBank + SVD Rank-40", "SVD", "file", "cnn_rank40"));
    return m;
}

double[][] methodMap(MethodSpec spec, double[][] crop01, Path filePath) throws Exception {
    if ("file".equals(spec.mode())) {
        if (!Files.isRegularFile(filePath)) return null;
        BufferedImage bi = ImageIO.read(filePath.toFile());
        if (bi == null) return null;
        return normalize01(gray01(bi));
    }
    if ("Gabor Transform".equals(spec.label())) return normalize01(gaborEnergy(crop01));
    if ("Chirplet Transform".equals(spec.label())) return normalize01(chirpletEnergy(crop01));
    if ("Morlet Wavelet".equals(spec.label())) return normalize01(morletEnergy(crop01));
    return null;
}

double mean(List<Double> xs) {
    if (xs.isEmpty()) return Double.NaN;
    double s = 0.0;
    for (double v : xs) s += v;
    return s / xs.size();
}

double std(List<Double> xs) {
    if (xs.size() < 2) return 0.0;
    double m = mean(xs);
    double s = 0.0;
    for (double v : xs) { double d = v - m; s += d * d; }
    return Math.sqrt(s / (xs.size() - 1));
}

String csvEscape(String s) {
    if (s == null) return "";
    if (s.contains(",") || s.contains("\"") || s.contains("\n")) {
        return "\"" + s.replace("\"", "\"\"") + "\"";
    }
    return s;
}

System.out.println("Unified scoring helpers loaded ✅");

Unified scoring helpers loaded ✅


In [3]:
// ---------- Build unified per-tile scoring table ----------
Path root = findProjectRoot(Paths.get(System.getProperty("user.dir")));
Path assets = root.resolve("notebooks").resolve("_assets");
Path waveletRoot = assets.resolve("fiba_wavelet_montage");
Path fftRoot = assets.resolve("fiba_tile_montage");

List<String> bases = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> waveletDirs = Arrays.asList(
    waveletRoot.resolve("generated_from_source"),
    waveletRoot.resolve("generated_from_source_picture1")
);
List<Path> fftDirs = Arrays.asList(
    fftRoot.resolve("generated_from_source"),
    fftRoot.resolve("generated_from_source_picture1")
);

Map<String, MethodSpec> methodSpecs = buildMethodSpecs();
List<UnifiedRow> rows = new ArrayList<>();
final double MASK_THR = 0.55;

for (int ds = 0; ds < bases.size(); ds++) {
    String base = bases.get(ds);
    Path waveletDir = waveletDirs.get(ds);
    Path fftDir = fftDirs.get(ds);
    if (!Files.isDirectory(waveletDir) || !Files.isDirectory(fftDir)) continue;

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> tileIds = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(waveletDir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) tileIds.add(Integer.parseInt(m.group(1)));
        }
    }
    tileIds.sort(Comparator.naturalOrder());

    for (int tileId : tileIds) {
        Path cropPath = waveletDir.resolve(base + "_tile" + tileId + "_crop.jpg");
        Path recPath = waveletDir.resolve(base + "_tile" + tileId + "_rec.jpg");
        if (!Files.isRegularFile(cropPath) || !Files.isRegularFile(recPath)) continue;

        BufferedImage cropBI = ImageIO.read(cropPath.toFile());
        BufferedImage recBI = ImageIO.read(recPath.toFile());
        if (cropBI == null || recBI == null) continue;

        double[][] crop01 = gray01(cropBI);
        double[][] rec01 = normalize01(gray01(recBI));

        double thetaRad = dominantThetaRad(crop01);
        double thetaDeg = Math.toDegrees(thetaRad);
        double[][] trajMask = trajectoryMask01(crop01.length, crop01[0].length, thetaRad);

        for (Map.Entry<String, MethodSpec> me : methodSpecs.entrySet()) {
            MethodSpec spec = me.getValue();
            Path filePath = null;
            if ("file".equals(spec.mode())) {
                if ("fft".equals(spec.fileKey())) {
                    filePath = fftDir.resolve(base + "_tile" + tileId + "_fft.jpg");
                } else {
                    filePath = waveletDir.resolve(base + "_tile" + tileId + "_" + spec.fileKey() + ".jpg");
                }
            }

            double[][] map01 = methodMap(spec, crop01, filePath);
            if (map01 == null) continue;

            double sep = maskSeparation(map01, trajMask, MASK_THR);
            double corrRaw = pearsonCenter(map01, rec01);
            double corr = Math.max(0.0, Math.min(1.0, 0.5 * (corrRaw + 1.0)));
            double score = 0.7 * sep + 0.3 * corr;

            rows.add(new UnifiedRow(
                waveletDir.getFileName().toString(),
                base,
                tileId,
                spec.label(),
                spec.family(),
                spec.mode(),
                thetaDeg,
                sep,
                corr,
                score
            ));
        }
    }
}

rows.sort(Comparator
    .comparing(UnifiedRow::datasetDir)
    .thenComparing(UnifiedRow::base)
    .thenComparingInt(UnifiedRow::tileId)
    .thenComparing(UnifiedRow::family)
    .thenComparing(UnifiedRow::method));

if (rows.isEmpty()) throw new RuntimeException("No unified scoring rows generated. Ensure source notebooks have produced assets.");
System.out.println("Unified per-tile rows collected: " + rows.size());

Unified per-tile rows collected: 140


In [4]:
// ---------- Aggregate unified score by dataset/method and family ----------
Map<String, List<UnifiedRow>> byDatasetMethod = new LinkedHashMap<>();
for (UnifiedRow r : rows) {
    String key = r.datasetDir() + "|" + r.family() + "|" + r.method();
    byDatasetMethod.computeIfAbsent(key, k -> new ArrayList<>()).add(r);
}

List<UnifiedSummary> methodSummary = new ArrayList<>();
for (Map.Entry<String, List<UnifiedRow>> e : byDatasetMethod.entrySet()) {
    String[] k = e.getKey().split("\\|", 3);
    String datasetDir = k[0], family = k[1], method = k[2];
    List<UnifiedRow> rs = e.getValue();

    List<Double> seps = new ArrayList<>();
    List<Double> corrs = new ArrayList<>();
    List<Double> scores = new ArrayList<>();
    List<Double> thetas = new ArrayList<>();

    for (UnifiedRow r : rs) {
        if (Double.isFinite(r.maskSeparation())) seps.add(r.maskSeparation());
        if (Double.isFinite(r.corrWithRec())) corrs.add(r.corrWithRec());
        if (Double.isFinite(r.unifiedScore())) scores.add(r.unifiedScore());
        if (Double.isFinite(r.trajectoryThetaDeg())) thetas.add(r.trajectoryThetaDeg());
    }

    methodSummary.add(new UnifiedSummary(
        datasetDir, family, method, rs.size(),
        mean(seps), std(seps),
        mean(corrs), std(corrs),
        mean(scores), std(scores),
        mean(thetas)
    ));
}
methodSummary.sort(Comparator
    .comparing(UnifiedSummary::datasetDir)
    .thenComparing(UnifiedSummary::family)
    .thenComparing((a, b) -> Double.compare(b.scoreMean(), a.scoreMean()))
    .thenComparing(UnifiedSummary::method));

Map<String, List<UnifiedRow>> byDatasetFamily = new LinkedHashMap<>();
for (UnifiedRow r : rows) {
    String key = r.datasetDir() + "|" + r.family();
    byDatasetFamily.computeIfAbsent(key, k -> new ArrayList<>()).add(r);
}

List<UnifiedSummary> familySummary = new ArrayList<>();
for (Map.Entry<String, List<UnifiedRow>> e : byDatasetFamily.entrySet()) {
    String[] k = e.getKey().split("\\|", 2);
    String datasetDir = k[0], family = k[1];
    List<UnifiedRow> rs = e.getValue();

    List<Double> seps = new ArrayList<>();
    List<Double> corrs = new ArrayList<>();
    List<Double> scores = new ArrayList<>();
    List<Double> thetas = new ArrayList<>();

    for (UnifiedRow r : rs) {
        if (Double.isFinite(r.maskSeparation())) seps.add(r.maskSeparation());
        if (Double.isFinite(r.corrWithRec())) corrs.add(r.corrWithRec());
        if (Double.isFinite(r.unifiedScore())) scores.add(r.unifiedScore());
        if (Double.isFinite(r.trajectoryThetaDeg())) thetas.add(r.trajectoryThetaDeg());
    }

    familySummary.add(new UnifiedSummary(
        datasetDir, family, "ALL", rs.size(),
        mean(seps), std(seps),
        mean(corrs), std(corrs),
        mean(scores), std(scores),
        mean(thetas)
    ));
}
familySummary.sort(Comparator.comparing(UnifiedSummary::datasetDir).thenComparing(UnifiedSummary::family));

System.out.println("Unified method-level summary rows: " + methodSummary.size());
String currentDataset = "";
for (UnifiedSummary s : methodSummary) {
    if (!s.datasetDir().equals(currentDataset)) {
        currentDataset = s.datasetDir();
        System.out.println("\nDataset: " + currentDataset);
        System.out.println("method\tfamily\tscore\tsep\tcorr\tnTiles");
    }
    System.out.printf(Locale.US, "%s\t%s\t%.5f\t%.5f\t%.5f\t%d%n",
        s.method(), s.family(), s.scoreMean(), s.sepMean(), s.corrMean(), s.nTiles());
}

System.out.println("\nUnified family-level summary (by dataset):");
for (UnifiedSummary s : familySummary) {
    System.out.printf(Locale.US, "%s | %s | score=%.5f | sep=%.5f | corr=%.5f | n=%d%n",
        s.datasetDir(), s.family(), s.scoreMean(), s.sepMean(), s.corrMean(), s.nTiles());
}

System.out.println("\nParity note:");
System.out.println("- FFT Pipeline uses generated *_fft.jpg from the FFT notebook outputs.");
System.out.println("- Gabor/Chirplet/Morlet are computed with the same formulas used in the wavelet comparison notebook.");
System.out.println("- SVD methods use generated *_svd_rank20.jpg, *_svd_energy99.jpg, *_cnn_rank40.jpg from the SVD notebook outputs.");

Unified method-level summary rows: 14

Dataset: generated_from_source
method	family	score	sep	corr	nTiles
FFT Pipeline	FFT	0.34297	0.27010	0.51299	10
Chirplet Transform	FFT-like/Wavelet	0.28456	0.02240	0.89625	10
Gabor Transform	FFT-like/Wavelet	0.28022	0.02718	0.87064	10
Morlet Wavelet	FFT-like/Wavelet	0.28022	0.02718	0.87064	10
SVD Energy-99%	SVD	0.29449	0.01538	0.94576	10
SVD Rank-20	SVD	0.29241	0.01532	0.93895	10
CNN-style ConvBank + SVD Rank-40	SVD	0.25555	0.02709	0.78863	10

Dataset: generated_from_source_picture1
method	family	score	sep	corr	nTiles
FFT Pipeline	FFT	0.36863	0.30836	0.50928	10
Gabor Transform	FFT-like/Wavelet	0.34090	0.11078	0.87783	10
Morlet Wavelet	FFT-like/Wavelet	0.34090	0.11078	0.87783	10
Chirplet Transform	FFT-like/Wavelet	0.33159	0.07588	0.92825	10
SVD Energy-99%	SVD	0.32353	0.05833	0.94234	10
SVD Rank-20	SVD	0.32043	0.05673	0.93573	10
CNN-style ConvBank + SVD Rank-40	SVD	0.29621	0.09130	0.77435	10

Unified family-level summary (by dataset):
generated_from_

In [7]:
// ---------- Export unified CSVs + score chart ----------
Path outDir = waveletRoot.resolve("master_analysis");
Files.createDirectories(outDir);

Path perTileCsv = outDir.resolve("master_unified_per_tile.csv");
Path methodCsv = outDir.resolve("master_unified_method_summary.csv");
Path familyCsv = outDir.resolve("master_unified_family_summary.csv");
Path chartPng = outDir.resolve("master_unified_family_comparison_java.png");

StringBuilder sbTile = new StringBuilder();
sbTile.append("dataset_dir,base,tile_id,method,family,source_kind,trajectory_theta_deg,mask_separation,corr_with_rec,unified_score\n");
for (UnifiedRow r : rows) {
    sbTile.append(csvEscape(r.datasetDir())).append(',')
        .append(csvEscape(r.base())).append(',')
        .append(r.tileId()).append(',')
        .append(csvEscape(r.method())).append(',')
        .append(csvEscape(r.family())).append(',')
        .append(csvEscape(r.sourceKind())).append(',')
        .append(String.format(Locale.US, "%.8f,%.8f,%.8f,%.8f%n", r.trajectoryThetaDeg(), r.maskSeparation(), r.corrWithRec(), r.unifiedScore()));
}
Files.writeString(perTileCsv, sbTile.toString(), StandardCharsets.UTF_8);

StringBuilder sbMethod = new StringBuilder();
sbMethod.append("dataset_dir,family,method,n_tiles,sep_mean,sep_std,corr_mean,corr_std,score_mean,score_std,theta_mean\n");
for (UnifiedSummary s : methodSummary) {
    sbMethod.append(csvEscape(s.datasetDir())).append(',')
        .append(csvEscape(s.family())).append(',')
        .append(csvEscape(s.method())).append(',')
        .append(s.nTiles()).append(',')
        .append(String.format(Locale.US, "%.8f,%.8f,%.8f,%.8f,%.8f,%.8f,%.8f%n",
            s.sepMean(), s.sepStd(),
            s.corrMean(), s.corrStd(),
            s.scoreMean(), s.scoreStd(), s.thetaMean()));
}
Files.writeString(methodCsv, sbMethod.toString(), StandardCharsets.UTF_8);

StringBuilder sbFamily = new StringBuilder();
sbFamily.append("dataset_dir,family,n_tiles,sep_mean,corr_mean,score_mean,theta_mean\n");
for (UnifiedSummary s : familySummary) {
    sbFamily.append(csvEscape(s.datasetDir())).append(',')
        .append(csvEscape(s.family())).append(',')
        .append(s.nTiles()).append(',')
        .append(String.format(Locale.US, "%.8f,%.8f,%.8f,%.8f%n",
            s.sepMean(), s.corrMean(), s.scoreMean(), s.thetaMean()));
}
Files.writeString(familyCsv, sbFamily.toString(), StandardCharsets.UTF_8);

int W = 1320, H = 440;
BufferedImage chart = new BufferedImage(W, H, BufferedImage.TYPE_INT_RGB);
Graphics2D g = chart.createGraphics();
g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
g.setColor(Color.WHITE);
g.fillRect(0, 0, W, H);

g.setColor(new Color(20, 40, 70));
g.setFont(new Font("SansSerif", Font.BOLD, 20));
g.drawString("Unified Family Comparison (score = 0.7*sep + 0.3*corr)", 20, 30);

List<String> datasets = Arrays.asList("generated_from_source", "generated_from_source_picture1");
List<String> famOrder = Arrays.asList("FFT", "FFT-like/Wavelet", "SVD");
Map<String, UnifiedSummary> famMap = new HashMap<>();
for (UnifiedSummary s : familySummary) famMap.put(s.datasetDir() + "|" + s.family(), s);

int panelY = 70, panelH = 320;
int panelW = (W - 80) / datasets.size();
for (int dsi = 0; dsi < datasets.size(); dsi++) {
    String dsName = datasets.get(dsi);
    int x0 = 20 + dsi * (panelW + 20);

    g.setColor(new Color(245, 247, 250));
    g.fillRect(x0, panelY, panelW, panelH);
    g.setColor(new Color(90, 100, 120));
    g.drawRect(x0, panelY, panelW, panelH);
    g.setColor(new Color(25, 25, 25));
    g.setFont(new Font("SansSerif", Font.BOLD, 14));
    g.drawString(dsName, x0 + 10, panelY + 20);

    double maxV = 1e-9;
    for (String fam : famOrder) {
        UnifiedSummary s = famMap.get(dsName + "|" + fam);
        if (s != null && Double.isFinite(s.scoreMean())) maxV = Math.max(maxV, s.scoreMean());
    }

    int barBaseY = panelY + panelH - 34;
    int barAvailH = panelH - 84;
    int barW = (panelW - 46) / famOrder.size() - 12;
    for (int i = 0; i < famOrder.size(); i++) {
        String fam = famOrder.get(i);
        UnifiedSummary s = famMap.get(dsName + "|" + fam);
        double v = (s == null || !Double.isFinite(s.scoreMean())) ? 0.0 : s.scoreMean();
        int hBar = (int)Math.round(v / maxV * barAvailH);
        int bx = x0 + 24 + i * (barW + 12);
        int by = barBaseY - hBar;

        Color col = ("FFT".equals(fam)) ? new Color(76, 120, 168) : ("SVD".equals(fam)) ? new Color(84, 162, 75) : new Color(245, 133, 24);
        g.setColor(col);
        g.fillRect(bx, by, barW, hBar);
        g.setColor(new Color(40, 40, 40));
        g.drawRect(bx, by, barW, hBar);

        g.setFont(new Font("SansSerif", Font.PLAIN, 11));
        g.drawString(fam, bx, barBaseY + 15);
        g.drawString(String.format(Locale.US, "%.3f", v), bx, Math.max(panelY + 40, by - 4));
    }
}

g.dispose();
ImageIO.write(chart, "png", chartPng.toFile());

System.out.println("Saved unified outputs:");
System.out.println(" - " + perTileCsv);
System.out.println(" - " + methodCsv);
System.out.println(" - " + familyCsv);
System.out.println(" - " + chartPng);

Saved unified outputs:
 - c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\master_analysis\master_unified_per_tile.csv
 - c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\master_analysis\master_unified_method_summary.csv
 - c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\master_analysis\master_unified_family_summary.csv
 - c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\master_analysis\master_unified_family_comparison_java.png


## How to use this master notebook (Java-only)

1. Run generation notebooks first so assets exist under:
   - `notebooks/_assets/fiba_tile_montage/...` (FFT maps)
   - `notebooks/_assets/fiba_wavelet_montage/...` (wavelet + SVD maps)
2. Use a **Java kernel** and run this notebook top-to-bottom.
3. Review outputs in `notebooks/_assets/fiba_wavelet_montage/master_analysis/`:
   - `master_unified_family_summary.csv`
   - `master_unified_method_summary.csv`
   - `master_unified_per_tile.csv`
   - `master_unified_family_comparison_java.png`

This notebook intentionally avoids Python and runs fully in Java.

## Broader background: why these techniques matter

### FFT family (frequency-domain directional analysis)

FFT-based transforms convert spatial image content into frequency coordinates, where oriented and periodic structures become directional energy concentrations. In fibrous tissues, this is often useful for rapidly detecting dominant alignment trends and coarse anisotropy signatures.

Practical image-analysis strengths:
- strong response to repeating, oriented microstructure,
- efficient global characterization,
- useful baseline for orientation tracking across regions.

Common biology applications:
- collagen orientation and remodeling trends,
- muscle/ECM alignment mapping,
- directional texture phenotyping in histology and microscopy.

### FFT-like / wavelet-style family (localized multiscale orientation)

Wavelet and filter-bank methods (including Gabor/Morlet-style responses) preserve locality in both space and frequency. This is valuable when tissue architecture is heterogeneous, with local domains of different alignment, waviness, or scale.

Practical image-analysis strengths:
- multiscale sensitivity to fine and coarse structure,
- local orientation maps rather than only global trends,
- robust interpretation when alignment varies spatially.

Common biology applications:
- local collagen disorder and microenvironment heterogeneity,
- subregion-specific directional signatures in tumor stroma,
- vessel/fiber neighborhood analysis where global FFT can average out detail.

### SVD / low-rank family (structure-noise separation)

SVD decomposes an image into dominant low-rank structure plus residual detail/noise-like components. In practice, low-rank reconstructions can stabilize metrics under noisy acquisition while keeping major morphology.

Practical image-analysis strengths:
- denoising-like behavior without heavy hand-tuned filtering,
- improved consistency of downstream angle/segmentation metrics,
- complementary to frequency methods when SNR is limited.

Common biology applications:
- microscopy denoising and robust quantification,
- preserving gross fiber architecture under noisy conditions,
- controlled tradeoff between detail retention and stability.

## Unified-scoring interpretation guide

### Unified objective

$$
\text{score}=0.7\cdot(\text{mask-separation})+0.3\cdot(\text{corr-with-rec})
$$

- **mask-separation**: transform intensity contrast inside vs outside the trajectory-angle mask.
- **corr-with-rec**: agreement with reconstruction image in center ROI, linearly mapped to $[0,1]$.

### Fairness/parity with original notebooks

- **FFT Pipeline** uses generated `*_fft.jpg` outputs from the FFT notebook.
- **Gabor/Chirplet/Morlet** are computed with the same formulas used in the wavelet notebook.
- **SVD Rank-20 / SVD Energy-99% / CNN Rank-40** use generated per-tile outputs from the SVD notebook.

### Suggested end-of-run summary template

- **Best method on dataset A:** `<method>` (`score=<value>`)
- **Best method on dataset B:** `<method>` (`score=<value>`)
- **Best family by dataset:** `<FFT / FFT-like/Wavelet / SVD>`
- **Biological interpretation:** `<1–2 sentence trajectory/organization interpretation>`

## Embedded unified comparison graphs

### Family-level unified comparison chart

![](_assets/fiba_wavelet_montage/master_analysis/master_unified_family_comparison_java.png)

### Data tables (open as CSV)

- `notebooks/_assets/fiba_wavelet_montage/master_analysis/master_unified_family_summary.csv`
- `notebooks/_assets/fiba_wavelet_montage/master_analysis/master_unified_method_summary.csv`
- `notebooks/_assets/fiba_wavelet_montage/master_analysis/master_unified_per_tile.csv`

In [6]:
// ---------- Build "cleanest-crop" full method panels for both datasets ----------

double laplacianSharpness(double[][] img01) {
    double[][] kl = new double[][] {{0,1,0},{1,-4,1},{0,1,0}};
    double[][] lap = convolveSame(img01, kl);
    int h = lap.length, w = lap[0].length;
    double s = 0.0;
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) s += lap[y][x] * lap[y][x];
    return s / Math.max(1, h * w);
}

BufferedImage toGrayImage01(double[][] a) {
    double[][] n = normalize01(a);
    int h = n.length, w = n[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * Math.max(0.0, Math.min(1.0, n[y][x])));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

BufferedImage colorizeTrajectoryMask(double[][] m) {
    int h = m.length, w = m[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = Math.max(0.0, Math.min(1.0, m[y][x]));
            int r = (int)Math.round(30 + 210 * v);
            int g = (int)Math.round(40 + 170 * v);
            int b = (int)Math.round(90 + 130 * v);
            out.setRGB(x, y, (r << 16) | (g << 8) | b);
        }
    }
    return out;
}

void drawFit(Graphics2D g, BufferedImage src, int x, int y, int w, int h) {
    double sx = w / (double) src.getWidth();
    double sy = h / (double) src.getHeight();
    double s = Math.min(sx, sy);
    int nw = Math.max(1, (int)Math.round(src.getWidth() * s));
    int nh = Math.max(1, (int)Math.round(src.getHeight() * s));
    int ox = x + (w - nw) / 2;
    int oy = y + (h - nh) / 2;
    g.drawImage(src, ox, oy, nw, nh, null);
}

BufferedImage makeMethodPanel(String title, List<String> labels, List<BufferedImage> imgs) {
    int cols = 4;
    int rows = (int)Math.ceil(labels.size() / (double)cols);
    int cellW = 250, cellH = 180, gap = 14;
    int margin = 20, top = 70, labelH = 18;
    int W = margin * 2 + cols * cellW + (cols - 1) * gap;
    int H = top + rows * (cellH + labelH + gap) + 24;

    BufferedImage canvas = new BufferedImage(W, H, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, W, H);

    g.setColor(new Color(18, 42, 76));
    g.setFont(new Font("SansSerif", Font.BOLD, 20));
    g.drawString(title, margin, 34);
    g.setFont(new Font("SansSerif", Font.PLAIN, 12));
    g.drawString("Panels: Crop, Reconstruction, Trajectory Mask, FFT, Gabor, Chirplet, Morlet, SVD variants", margin, 54);

    for (int i = 0; i < labels.size(); i++) {
        int r = i / cols;
        int c = i % cols;
        int x = margin + c * (cellW + gap);
        int y = top + r * (cellH + labelH + gap);

        g.setColor(new Color(40, 40, 40));
        g.drawString(labels.get(i), x + 2, y + 12);
        int iy = y + labelH;
        g.setColor(new Color(232, 232, 232));
        g.fillRect(x - 1, iy - 1, cellW + 2, cellH + 2);
        drawFit(g, imgs.get(i), x, iy, cellW, cellH);
    }

    g.dispose();
    return canvas;
}

Path panelOutDir = waveletRoot.resolve("master_analysis");
Files.createDirectories(panelOutDir);

for (int ds = 0; ds < bases.size(); ds++) {
    String base = bases.get(ds);
    Path waveletDir = waveletDirs.get(ds);
    Path fftDir = fftDirs.get(ds);
    if (!Files.isDirectory(waveletDir) || !Files.isDirectory(fftDir)) continue;

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> tileIds = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(waveletDir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) tileIds.add(Integer.parseInt(m.group(1)));
        }
    }
    tileIds.sort(Comparator.naturalOrder());
    if (tileIds.isEmpty()) continue;

    int bestTile = -1;
    double bestSharp = Double.NEGATIVE_INFINITY;
    for (int tileId : tileIds) {
        Path cropPath = waveletDir.resolve(base + "_tile" + tileId + "_crop.jpg");
        if (!Files.isRegularFile(cropPath)) continue;
        BufferedImage cropBI = ImageIO.read(cropPath.toFile());
        if (cropBI == null) continue;
        double s = laplacianSharpness(gray01(cropBI));
        if (s > bestSharp) {
            bestSharp = s;
            bestTile = tileId;
        }
    }
    if (bestTile < 0) continue;

    Path cropPath = waveletDir.resolve(base + "_tile" + bestTile + "_crop.jpg");
    Path recPath = waveletDir.resolve(base + "_tile" + bestTile + "_rec.jpg");
    Path fftPath = fftDir.resolve(base + "_tile" + bestTile + "_fft.jpg");
    Path r20Path = waveletDir.resolve(base + "_tile" + bestTile + "_svd_rank20.jpg");
    Path e99Path = waveletDir.resolve(base + "_tile" + bestTile + "_svd_energy99.jpg");
    Path cnnPath = waveletDir.resolve(base + "_tile" + bestTile + "_cnn_rank40.jpg");

    BufferedImage cropBI = ImageIO.read(cropPath.toFile());
    BufferedImage recBI = Files.isRegularFile(recPath) ? ImageIO.read(recPath.toFile()) : null;
    BufferedImage fftBI = Files.isRegularFile(fftPath) ? ImageIO.read(fftPath.toFile()) : null;
    BufferedImage r20BI = Files.isRegularFile(r20Path) ? ImageIO.read(r20Path.toFile()) : null;
    BufferedImage e99BI = Files.isRegularFile(e99Path) ? ImageIO.read(e99Path.toFile()) : null;
    BufferedImage cnnBI = Files.isRegularFile(cnnPath) ? ImageIO.read(cnnPath.toFile()) : null;
    if (cropBI == null) continue;

    double[][] crop01 = gray01(cropBI);
    double theta = dominantThetaRad(crop01);
    double[][] trajMask = trajectoryMask01(crop01.length, crop01[0].length, theta);

    BufferedImage gaborBI = toGrayImage01(gaborEnergy(crop01));
    BufferedImage chirpBI = toGrayImage01(chirpletEnergy(crop01));
    BufferedImage morletBI = toGrayImage01(morletEnergy(crop01));

    List<String> labels = new ArrayList<>();
    List<BufferedImage> imgs = new ArrayList<>();

    labels.add("Crop (cleanest tile=" + bestTile + ")"); imgs.add(cropBI);
    labels.add("Reconstruction"); imgs.add(recBI != null ? recBI : toGrayImage01(crop01));
    labels.add("Trajectory-angle mask"); imgs.add(colorizeTrajectoryMask(trajMask));
    labels.add("FFT Pipeline"); imgs.add(fftBI != null ? fftBI : toGrayImage01(crop01));
    labels.add("Gabor Transform"); imgs.add(gaborBI);
    labels.add("Chirplet Transform"); imgs.add(chirpBI);
    labels.add("Morlet Wavelet"); imgs.add(morletBI);
    labels.add("SVD Rank-20"); imgs.add(r20BI != null ? r20BI : toGrayImage01(crop01));
    labels.add("SVD Energy-99%"); imgs.add(e99BI != null ? e99BI : toGrayImage01(crop01));
    labels.add("CNN + SVD Rank-40"); imgs.add(cnnBI != null ? cnnBI : toGrayImage01(crop01));

    BufferedImage panel = makeMethodPanel(
        "Full method panel: " + base + " (cleanest crop by Laplacian sharpness)",
        labels,
        imgs
    );

    Path outPng = panelOutDir.resolve(base + "_cleanest_full_method_panel.png");
    ImageIO.write(panel, "png", outPng.toFile());

    System.out.printf(Locale.US,
        "%s cleanest tile=%d sharpness=%.6f panel=%s%n",
        base, bestTile, bestSharp, outPng
    );
}

System.out.println("Cleanest-crop full panels generated ✅");

C15D5P001_1 cleanest tile=10 sharpness=0.028590 panel=c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\master_analysis\C15D5P001_1_cleanest_full_method_panel.png
Picture1 cleanest tile=1 sharpness=0.051027 panel=c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\master_analysis\Picture1_cleanest_full_method_panel.png
Cleanest-crop full panels generated ✅


## Cleanest-crop full panels (both image sources)

These panels are automatically chosen from the **cleanest crop tile** in each source image using Laplacian-based sharpness, then expanded to show all major method outputs side-by-side.

### Source A (`C15D5P001_1`) cleanest crop full panel

![](_assets/fiba_wavelet_montage/master_analysis/C15D5P001_1_cleanest_full_method_panel.png)

### Source B (`Picture1`) cleanest crop full panel

![](_assets/fiba_wavelet_montage/master_analysis/Picture1_cleanest_full_method_panel.png)

### Unified graph quick view

![](_assets/fiba_wavelet_montage/master_analysis/master_unified_family_comparison_java.png)

If a preview does not render in your notebook viewer, open the PNG files directly from `notebooks/_assets/fiba_wavelet_montage/master_analysis/`.